# Module 4 · Real-Time Eventing & AI Readiness

**Story:** two capabilities in one notebook —

- **Part A** — a Change Stream reacts to a wire transfer flipping from
  *Pending* to *Executed* in near-real-time, the same mechanism that powers
  Atlas Triggers (a managed, hosted version of this same listener that can
  call a webhook/Lambda without you running the process yourself).
- **Part B** — Vector Search over transaction memo text, for categorization
  and anomaly-detection style queries ("find transactions that look like
  this one"), using transaction descriptions as the example corpus.

Embeddings here are generated **locally** with `sentence-transformers` — no
external API key needed, so this notebook runs standalone. In production,
MongoDB's recommended path is **Voyage AI** embeddings (native integration,
no self-hosted model to manage) — swap the embedding call for a Voyage API
call and everything else in this notebook is unchanged.


In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pymongo[encryption]", "certifi", "python-dotenv", "requests", "matplotlib", "pandas", "sentence-transformers"],
        check=True,
    )
    from getpass import getpass
    ATLAS_URI = os.environ.get("ATLAS_URI") or getpass("Atlas connection string (ATLAS_URI): ")
else:
    from dotenv import load_dotenv
    load_dotenv()
    ATLAS_URI = os.environ["ATLAS_URI"]

DEMO_DB = os.environ.get("DEMO_DB", "amex_demo")
import certifi
CA_FILE = certifi.where()
print("Environment:", "Colab" if IN_COLAB else "local", "| DB:", DEMO_DB)

import random
from datetime import datetime, timedelta, timezone
from pymongo import MongoClient

client = MongoClient(ATLAS_URI, tlsCAFile=CA_FILE)
db = client[DEMO_DB]
movement = db.money_movement

# Seed a small batch if Module 2 hasn't been run yet, so this notebook works standalone.
if movement.count_documents({}) == 0:
    random.seed(11)
    seed_docs = [
        {
            "transfer_id": f"MM{i:09d}",
            "type": random.choice(["wire", "ach", "bill_pay"]),
            "status": "Pending",
            "direction": random.choice(["incoming", "outgoing"]),
            "amount": round(random.uniform(500, 20000), 2),
            "created_at": datetime.now(timezone.utc) - timedelta(minutes=random.randint(0, 500)),
        }
        for i in range(200)
    ]
    movement.insert_many(seed_docs)
    print("Seeded 200 pending transfers (Module 2 wasn't run first).")

print("money_movement docs:", movement.count_documents({}))


## Part A — Change Streams: react to a status change in real time

Watch for updates where `status` transitions to `"Executed"`. In another
thread, simulate a few wire transfers clearing.


In [ ]:
import threading, time

def clear_transfers():
    time.sleep(3)
    pending_ids = [
        d["_id"] for d in movement.find({"status": "Pending"}).limit(5)
    ]
    for _id in pending_ids:
        time.sleep(1.5)
        movement.update_one({"_id": _id}, {"$set": {"status": "Executed"}})

clearer = threading.Thread(target=clear_transfers, daemon=True)
clearer.start()

change_pipeline = [
    {"$match": {
        "operationType": "update",
        "updateDescription.updatedFields.status": "Executed",
    }}
]

print("Watching for transfers clearing... (about 12 seconds)")
deadline = time.time() + 12
with movement.watch(pipeline=change_pipeline, full_document="updateLookup") as stream:
    while time.time() < deadline:
        change = stream.try_next()
        if change is None:
            time.sleep(0.2)
            continue
        doc = change["fullDocument"]
        print(f"[notify] {doc['transfer_id']} ({doc['type']}, ${doc['amount']:,.2f}) — Executed")

clearer.join(timeout=2)
print("Done.")


This is exactly the mechanism behind **Atlas Triggers** — same change
stream, except Atlas hosts the listener and calls your webhook/Lambda/Slack
notification for you, rather than you running this Python process.

## Part B — Vector Search for transaction categorization / anomaly detection

Seed a small, curated set of transaction memos — mostly routine, a couple
written to look unusual — embed them locally, and run a semantic search.


In [ ]:
memos_coll = db.transaction_memos

sample_memos = [
    {"memo": "Wire transfer to vendor for office equipment purchase", "amount": 4200.00},
    {"memo": "ACH payment - monthly office supplies order", "amount": 380.15},
    {"memo": "Bill pay - commercial lease payment", "amount": 12500.00},
    {"memo": "Internal transfer between corporate operating accounts", "amount": 50000.00},
    {"memo": "Payroll disbursement - biweekly run", "amount": 88000.00},
    {"memo": "Wire transfer - vendor payment for IT services", "amount": 6100.00},
    {"memo": "ACH refund to customer for returned merchandise", "amount": 120.00},
    {"memo": "Bill pay - utility payment, electric", "amount": 940.00},
    {"memo": "Large same-day wire to a newly added, unverified payee", "amount": 97500.00},
    {"memo": "Unusual late-night transfer to a personal account, first time recipient", "amount": 42000.00},
    {"memo": "Multiple rapid small transfers to different new external accounts", "amount": 950.00},
    {"memo": "Wire transfer for equipment lease buyout", "amount": 15300.00},
    {"memo": "ACH payment - insurance premium", "amount": 2100.00},
    {"memo": "Bill pay - vendor invoice, office cleaning services", "amount": 610.00},
    {"memo": "Internal transfer to fund new branch operating account", "amount": 75000.00},
]

memos_coll.drop()
memos_coll.insert_many(sample_memos)
print(f"Inserted {memos_coll.count_documents({})} transaction memos.")


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # 384-dim, runs on CPU, no API key

docs = list(memos_coll.find({}))
embeddings = model.encode([d["memo"] for d in docs], show_progress_bar=False)

for doc, emb in zip(docs, embeddings):
    memos_coll.update_one({"_id": doc["_id"]}, {"$set": {"embedding": emb.tolist()}})

print("Embeddings generated and stored. Dimensions:", len(embeddings[0]))


In [ ]:
from pymongo.operations import SearchIndexModel
import time

search_index_model = SearchIndexModel(
    definition={
        "fields": [
            {"type": "vector", "path": "embedding", "numDimensions": 384, "similarity": "cosine"},
        ]
    },
    name="memo_vector_index",
    type="vectorSearch",
)

memos_coll.create_search_index(model=search_index_model)

print("Waiting for the vector index to build...")
while True:
    idx = next((i for i in memos_coll.list_search_indexes() if i["name"] == "memo_vector_index"), None)
    if idx and idx.get("status") == "READY" and idx.get("queryable"):
        break
    time.sleep(3)
print("Index ready.")


In [ ]:
query_text = "unusual large payment to an unfamiliar recipient"
query_vector = model.encode(query_text).tolist()

pipeline = [
    {"$vectorSearch": {
        "index": "memo_vector_index",
        "path": "embedding",
        "queryVector": query_vector,
        "numCandidates": 50,
        "limit": 5,
    }},
    {"$project": {
        "_id": 0, "memo": 1, "amount": 1,
        "score": {"$meta": "vectorSearchScore"},
    }},
]

print(f"Query: {query_text!r}\n")
for r in memos_coll.aggregate(pipeline):
    print(f"  {r['score']:.3f}  ${r['amount']:>10,.2f}  {r['memo']}")


### Presenter talking points

- Change Streams → Atlas Triggers is the direct answer to "delayed
  communications" — status changes reach downstream systems in
  milliseconds, not on a polling schedule.
- Vector Search ranks by **meaning**, not keywords — notice the top results
  are semantically about unusual/unfamiliar-recipient payments even though
  the query text doesn't share exact wording with every matching memo.
- This is the same `$vectorSearch` operator and index type used for RAG/AI
  agent use cases — the "AI readiness" story is that this capability is
  already sitting on the same cluster as the operational data, not a
  separate system to stand up and keep in sync.
- Production path: swap the local `sentence-transformers` call for a Voyage
  AI embedding call (or Atlas's built-in automatic embedding) — same index,
  same query shape.
